# MedNorm-VI — VietMed-NER Parquet Preprocessing (Colab CPU)

Real, executable workflow for a **fresh Colab CPU runtime**. Not model training; no GPU.

`RUN_MODE = "REAL"` preprocesses the real VietMed Parquet from Drive.
`RUN_MODE = "SYNTHETIC_SMOKE"` exercises the same code path on tiny synthetic rows
(used by the repository's behavioral test) and is **never** proof of real preprocessing.


## 1. Runtime detection (Colab, Python, RAM, disk)


In [ ]:
import os
import sys
import platform
import shutil

IN_COLAB = 'google.colab' in sys.modules
_total, _used, _free = shutil.disk_usage('/')
print('python', platform.python_version())
print('in_colab', IN_COLAB)
print('disk_free_gb', round(_free / 1e9, 1))
try:
    import multiprocessing
    print('cpu_count', multiprocessing.cpu_count())
except Exception as exc:
    print('cpu_count unavailable:', exc)


## 2. Execution mode (REAL by default; no silent fallback)


In [ ]:
RUN_MODE = os.environ.get('MEDNORM_RUN_MODE', 'REAL')
assert RUN_MODE in ('REAL', 'SYNTHETIC_SMOKE'), 'RUN_MODE must be REAL or SYNTHETIC_SMOKE'
if RUN_MODE == 'SYNTHETIC_SMOKE':
    print('*** SYNTHETIC_SMOKE: tiny synthetic rows only. NOT real VietMed preprocessing. ***')
else:
    print('REAL mode: real VietMed Parquet required; missing data will fail fast.')


## 3. Mount Google Drive (REAL mode only)


In [ ]:
if RUN_MODE == 'REAL' and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('drive mounted')
else:
    print('drive mount skipped (mode=' + RUN_MODE + ', in_colab=' + str(IN_COLAB) + ')')


## 4. Configuration — edit only these three values
`DRIVE_ROOT` is persistent Drive storage; `REPO_DIR` is a temporary Colab checkout.
All other paths are derived automatically.


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path(os.environ.get('MEDNORM_DRIVE_ROOT', '/content/drive/MyDrive/MedNorm-VI'))
REPO_URL = os.environ.get('MEDNORM_REPO_URL', 'https://github.com/vquclinh/MedNorm-VI.git')
REPO_REF = os.environ.get('MEDNORM_REPO_REF', 'main')

REPO_DIR = Path(os.environ.get('MEDNORM_REPO_DIR', '/content/MedNorm-VI'))
VIETMED_SOURCE_DIR = DRIVE_ROOT / 'data' / 'external' / 'public_ner' / 'vietmed_ner'
VIETMED_PARQUET_DIR = VIETMED_SOURCE_DIR / 'data'
ARTIFACT_DIR = DRIVE_ROOT / 'data' / 'derived' / 'training_corpora' / 'vietmed_ner_v1'
print('DRIVE_ROOT        ', DRIVE_ROOT)
print('REPO_URL          ', REPO_URL)
print('REPO_REF          ', REPO_REF)
print('REPO_DIR          ', REPO_DIR)
print('VIETMED_PARQUET_DIR', VIETMED_PARQUET_DIR)
print('ARTIFACT_DIR      ', ARTIFACT_DIR)


## 5. Install pinned CPU dependency (executed, not commented)


In [ ]:
PYARROW_PIN = 'pyarrow==17.0.0'
if RUN_MODE == 'REAL':
    try:
        import pyarrow
        print('pyarrow already present', pyarrow.__version__)
    except ImportError:
        import subprocess
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', PYARROW_PIN], check=True)
        import pyarrow
        print('pyarrow installed', pyarrow.__version__)
else:
    print('SYNTHETIC_SMOKE: pyarrow not required (no Parquet is read)')


## 6. Real repository checkout (executed git clone/fetch/checkout)
Uses the tracked, tested helper `mednorm_vi.reproducibility.repository_checkout`.
It fails fast on placeholder values and verifies `src/mednorm_vi` and the adapter exist.


In [ ]:
import subprocess

# Bootstrap: obtain the checkout helper, then perform the real verified checkout.
_helper_src = os.environ.get('MEDNORM_CHECKOUT_HELPER_SRC', '')
if _helper_src and Path(_helper_src).is_dir():
    # Local/test bootstrap: a src/ directory that already contains mednorm_vi.
    if _helper_src not in sys.path:
        sys.path.insert(0, _helper_src)
else:
    # Colab: shallow-clone once to obtain the helper, then do the verified checkout.
    _bootstrap = Path('/content/_mednorm_bootstrap')
    if _bootstrap.exists():
        shutil.rmtree(_bootstrap)
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(_bootstrap)], check=True)
    if str(_bootstrap / 'src') not in sys.path:
        sys.path.insert(0, str(_bootstrap / 'src'))

from mednorm_vi.reproducibility.repository_checkout import checkout_repository  # noqa: E402

checkout = checkout_repository(REPO_URL, REPO_DIR, REPO_REF, clean_existing=True)
RESOLVED_COMMIT = checkout.resolved_commit
print('requested_ref  ', checkout.requested_ref)
print('resolved_commit', RESOLVED_COMMIT)
print('repo_dir       ', checkout.repo_dir)
print('src_dir        ', checkout.src_dir)
print('adapter_file   ', checkout.adapter_file)


## 7. Verify checkout and import the adapter from it


In [ ]:
assert Path(checkout.src_dir).is_dir(), 'src dir missing after checkout'
assert Path(checkout.adapter_file).is_file(), 'adapter file missing after checkout'
if checkout.src_dir not in sys.path:
    sys.path.insert(0, checkout.src_dir)
from mednorm_vi.data_engine import vietmed_ner as vm  # noqa: E402
print('adapter imported:', vm.ADAPTER_VERSION)
print('adapter module  :', vm.__file__)


## 8. Mapping file (from the verified checkout)


In [ ]:
MAPPING_PATH = Path(checkout.repo_dir) / vm.DEFAULT_MAPPING
assert MAPPING_PATH.is_file(), 'mapping file missing: ' + str(MAPPING_PATH)
mapping = vm.load_vietmed_mapping(MAPPING_PATH)
print('mapping version', mapping.version, 'concrete', mapping.type_mapping)
print('mapping config_hash', mapping.config_hash[:16])


## 9. Source discovery (REAL) / synthetic rows (SMOKE)


In [ ]:
import hashlib

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

if RUN_MODE == 'REAL':
    assert VIETMED_PARQUET_DIR.is_dir(), 'missing Parquet dir: ' + str(VIETMED_PARQUET_DIR)
    parquets = sorted(p.name for p in VIETMED_PARQUET_DIR.glob('*.parquet'))
    assert parquets, 'no .parquet files found (fail fast)'
    source_hashes = {f: sha256_file(VIETMED_PARQUET_DIR / f) for f in parquets}
    print('parquet files', parquets)
else:
    parquets = []
    source_hashes = {'SYNTHETIC': 'synthetic-smoke-no-source-file'}
    print('synthetic smoke: no Parquet source is read')


## 10. Column resolution (REAL: from the real Parquet schema)


In [ ]:
if RUN_MODE == 'REAL':
    import pyarrow.parquet as pq  # noqa: E402
    schema = pq.read_schema(str(VIETMED_PARQUET_DIR / parquets[0]))
    column_types = {f.name: str(f.type) for f in schema}
else:
    column_types = {'words': 'list<element: string>', 'tags': 'list<element: int64>',
                    'labels': 'list<element: string>', 'text': 'string'}
word_col, tag_col = vm.resolve_bio_columns(column_types)
resolved_columns = {'word_col': word_col, 'tag_col': tag_col}
print('all columns    ', sorted(column_types))
print('resolved columns', resolved_columns)


## 11. Read rows (words + BIO-string labels only; audio never read)


In [ ]:
if RUN_MODE == 'REAL':
    rows = vm.read_vietmed_parquet(VIETMED_PARQUET_DIR)
else:
    rows = [
        {'words': ['uong', 'paracetamol', '500mg'],
         'labels': ['O', 'B-DRUGCHEMICAL', 'I-DRUGCHEMICAL'],
         'split': 'train', 'record_id': '000001'},
        {'words': ['benh', 'nhan', 'ho'], 'labels': ['O', 'O', 'O'],
         'split': 'validation', 'record_id': '000002'},
    ]
assert rows, 'no rows read (fail fast)'
print('rows read', len(rows))


## 12. Convert to canonical half-open examples


In [ ]:
examples, summary = vm.convert_rows(rows, mapping=mapping)
print('examples ', summary['examples_emitted'])
print('entities ', summary['entities_emitted'])
print('accepted ', summary['accepted'], '| repaired', summary['deterministic_repair'],
      '| excluded', summary['excluded'], '| human_review', summary['human_review_required'])


## 13. Fail-fast integrity assertions


In [ ]:
assert summary['offset_invalid'] == 0, 'offset invariant violated (fail fast)'
assert summary['human_review_required'] == 0, 'words/tags mismatch needs human review (fail fast)'
for _ex in examples:
    for _en in _ex['entities']:
        assert _ex['text'][_en['start']:_en['end']] == _en['text'], 'span mismatch'
print('offset_invalid 0; human_review 0; all spans verified:', len(examples), 'examples')


## 14. Deterministic repeated conversion


In [ ]:
examples_again, _summary_again = vm.convert_rows(rows, mapping=mapping)
assert examples == examples_again, 'conversion is not deterministic (fail fast)'
print('deterministic rerun verified')


## 15. Write real artifacts (records the resolved commit SHA)


In [ ]:
manifest = vm.write_artifacts(
    ARTIFACT_DIR, examples, summary, mapping=mapping,
    source_hashes=source_hashes, repo_commit=RESOLVED_COMMIT,
    resolved_columns=resolved_columns)
print('repo_commit in manifest', manifest['repo_commit'])
print('examples_jsonl_sha256  ', manifest['examples_jsonl_sha256'])


## 16. Reload artifacts and verify manifest hashes


In [ ]:
reloaded = vm.load_vietmed_artifacts(ARTIFACT_DIR)
assert reloaded is not None, 'artifacts not found after write (fail fast)'
assert reloaded == examples, 'artifact roundtrip mismatch (fail fast)'
print('artifact manifest hash verified; rows', len(reloaded))


## 17. Verify no audio field or bytes in the emitted JSONL


In [ ]:
_jsonl_path = ARTIFACT_DIR / 'canonical_examples' / 'vietmed_ner_examples.jsonl'
_jsonl_text = _jsonl_path.read_text(encoding='utf-8')
assert 'audio' not in _jsonl_text, 'audio leaked into JSONL (fail fast)'
assert manifest['audio_excluded'] is True
print('audio exclusion verified')


## 18. Real artifact tree (paths, sizes, SHA-256)


In [ ]:
_expected = [
    'canonical_examples/vietmed_ner_examples.jsonl',
    'manifests/vietmed_ner_preprocessing_manifest.json',
    'manifests/vietmed_ner_source_hashes.json',
    'quality/vietmed_ner_label_inventory.json',
    'quality/vietmed_ner_offset_validation.json',
    'quality/vietmed_ner_repair_exclusion_summary.json',
    'quality/vietmed_ner_split_summary.json',
]
for _rel in _expected:
    _p = ARTIFACT_DIR / _rel
    assert _p.is_file(), 'missing expected artifact: ' + _rel
    print(round(_p.stat().st_size / 1024, 2), 'KB', sha256_file(_p)[:16], _rel)
print('artifact tree verified:', len(_expected), 'files')


## 19. Return to repository

1. Copy `ARTIFACT_DIR` (on Drive) into the repository at
   `data/derived/training_corpora/vietmed_ner_v1/` (git-ignored).
2. Rebuild the governed corpus locally:

```bash
env PYTHONPATH=src python3 -m mednorm_vi.data_engine.cli build-governed-corpus
```

Expect `vietmed_status: included_from_artifacts`, `cross_split_family_leakage 0`,
`eval_map_approximate_entities 0`. No raw Parquet, pyarrow, or audio is needed afterwards.

SYNTHETIC_SMOKE output is test-only and must never be returned as real preprocessing.
